# Aircraft Classification Demo — 5 Models

This notebook is the legacy training and inference workflow for a small aircraft image classification demo limited to **five aircraft classes**.

The purpose of this cleaned version is:
- to preserve the original code logic,
- to remove visual clutter such as stale outputs,
- to document each part of the workflow in clear English,
- to make the notebook easier to review and maintain.

## Scope

This notebook is intentionally lightweight.  
It is separate from the larger FGVC-Aircraft training pipeline used for the broader manufacturer and family experiments.

## Main workflow

The notebook follows a standard image-classification sequence:
1. import dependencies,
2. locate the dataset,
3. build the FastAI data pipeline,
4. define the model,
5. train the model,
6. export the trained artifact,
7. run a few inference checks.

## Important note

The code itself has **not been redesigned**.  
This version focuses on readability, structure, and documentation while preserving the original intent of the notebook.


## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
from fastcore.all import *
from fastai.vision.all import *
from fastai.vision.widgets import *
from fastdownload import download_url

## Notebook step

This section contains part of the original workflow and is documented here to make the legacy notebook easier to read.

In [ ]:
# -----------------------------------------------------------------------------
# NOTEBOOK STEP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
pip install ddgs

## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
from fastcore.all import *
from fastai.vision.all import *
from fastai.vision.widgets import *
from fastdownload import download_url

## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
from ddgs import DDGS
from pathlib import Path
from time import sleep
from tqdm import tqdm

from fastai.vision.all import *
from fastai.vision.widgets import *
from ddgs import DDGS

def search_images(query, max_results=30):
    with DDGS(timeout=30) as ddgs:
        results = ddgs.images(
            query,
            max_results=max_results,
            safesearch="off",
            region="wt-wt"
        )
    return [r["image"] for r in results if "image" in r]

avions = ['Dassault Rafale', 'Piper Cub', 'Boeing 737', 'McDonnell Douglas C-17', 'Citation XL']
path = Path('avions')

for b in tqdm(avions):
    dest = path / b
    dest.mkdir(exist_ok=True, parents=True)

    download_images(dest, urls=search_images(f'photo {b}'))
    sleep(10)

    download_images(dest, urls=search_images(f'photo {b} from above'))
    sleep(10)

    download_images(dest, urls=search_images(f'photo {b} from below'))
    sleep(10)

    resize_images(dest, max_size=400)

## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
from fastai.vision.utils import verify_images

fns = get_image_files(path)
failed = verify_images(fns)
failed

## Notebook step

This section contains part of the original workflow and is documented here to make the legacy notebook easier to read.

In [ ]:
# -----------------------------------------------------------------------------
# NOTEBOOK STEP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
failed.map(Path.unlink)

## Data pipeline construction

This section builds the FastAI data pipeline, including image loading, labels, transforms, and the training / validation split.

In [ ]:
# -----------------------------------------------------------------------------
# DATA PIPELINE CONSTRUCTION
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
# Build the FastAI DataLoaders object.
dls = DataBlock(
    blocks = (ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed = 18),
    get_y=parent_label,
    item_tfms=[Resize(256, method='squish')]
).dataloaders(path, bs=32)

## Data pipeline construction

This section builds the FastAI data pipeline, including image loading, labels, transforms, and the training / validation split.

In [ ]:
# -----------------------------------------------------------------------------
# DATA PIPELINE CONSTRUCTION
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
# Build the FastAI DataLoaders object.
dls = ImageDataLoaders.from_folder(path, valid_pct=0.2, seed=42, item_tfms=Resize(224))
# Create the learner object used for training and inference.
learn = vision_learner(dls, resnet34, metrics=accuracy)
# Train the model using FastAI fine-tuning.
learn.fine_tune(5)

## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
from fastai.vision.all import ClassificationInterpretation

interp = ClassificationInterpretation.from_learner(learn)

interp.plot_confusion_matrix(normalize = True)

## Model export

This section saves the trained artifact so it can be reused later for inference or deployment.

In [ ]:
# -----------------------------------------------------------------------------
# MODEL EXPORT
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
# Export the trained model for later reuse.
learn.export('airplane_recognition_model.pkl')

## Imports and environment setup

This section imports the libraries used in the notebook and prepares the execution environment.

In [ ]:
# -----------------------------------------------------------------------------
# IMPORTS AND ENVIRONMENT SETUP
# -----------------------------------------------------------------------------
# This code cell comes from the original legacy notebook.
# The logic is preserved, and comments have been added to make the role of the
# cell explicit within the end-to-end workflow.
!pip -q install torchview graphviz

from torchview import draw_graph
import torch

model = learn.model.eval()
graph = draw_graph(model, input_size=(1,3,224,224), expand_nested=True)
graph.visual_graph.render(
    filename="model_graph",
    format="png",
    cleanup=True
)

graph.visual_graph